# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, focusing on robust, reproducible handling of Croissant schemas.

### Dataset Source
The dataset is defined and described via a [Croissant schema](https://mlcommons.org/croissant/) accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata describing the dataset and explore the available record sets using `mlcroissant`. This connects to the FAIR^2 Croissant schema and provides an interface to its rich metadata and records.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nPublished: {metadata.datePublished}")

## 2. Data Overview

List and review all available record sets, fields, and their Croissant `@id` fields. This helps in identifying the structure of data before extraction. All references use the unique `@id`.

In [ ]:
# Retrieve and display all record sets and their fields by @id
print("Available record sets:")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_sets.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field: {field.name}      @id: {field.id}    Type: {field.data_type}")
    print()
# If you want to manually inspect a specific record set structure by @id:
# for x in dataset.records(record_set='<@id>'):
#     print(x)


## 3. Data Extraction
Load records for each record set into Pandas DataFrames using only `@id` fields. This enables flexible, robust access and referencing as the dataset evolves.

In [ ]:
# Extract data from all record sets, using their @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set: {record_set_id}")
    print(f"Columns: {list(dataframes[record_set_id].columns)}\n")
# Display first few rows of first available record set for illustration:
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"Preview of record set {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate filtering, normalization, and grouping using only `@id` references. Choose a numeric field and a group field for demonstration (replace these IDs as needed based on the output above or specific knowledge of the dataset schema).

> **Note:** You should replace `numeric_field_id` and `group_field_id` with actual `@id` values for your dataset. Here we use the first applicable numeric and group fields as an example.

In [ ]:
# Identify a numeric field and a group field from the first record set
df = dataframes[example_record_set_id]
numeric_field_id = None
group_field_id = None

# Attempt to auto-detect types (replace with specific IDs if required)
fields = {field.id: field for field in dataset.record_set(example_record_set_id).fields}
for field_id, field in fields.items():
    if field.data_type in ['Float', 'Integer', 'Number'] and field_id in df.columns and numeric_field_id is None:
        numeric_field_id = field_id
    if field.data_type in ['Text', 'String'] and field_id in df.columns and group_field_id is None:
        group_field_id = field_id

print(f"Numeric field selected for filtering (by @id): {numeric_field_id}")
print(f"Group field selected (by @id): {group_field_id}\n")

# Filter: Example threshold (customize as appropriate for your actual field)
if numeric_field_id:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Records in {example_record_set_id} where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a group field, if present
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped means by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields. We'll use the previously selected numeric and group fields. You can further customize visualizations to fit your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if group_field_id and numeric_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Connect to and explore a dataset defined by a Croissant schema using `mlcroissant`, referencing all entities by their `@id` fields.
- Inspect record sets and field structures.
- Extract records into DataFrames using record set `@id`.
- Filter, normalize, group, and visualize data fields by referencing `@id`, supporting reproducibility and future-proof pipelines.

**Next steps:**
- Adapt the data extraction and EDA steps to specific fields relevant to your domain question.
- Explore linked metadata (authors, funding, spatial/temporal coverage, etc.) using the rich Croissant-structured dataset.